# Prequential Evaluation — Function 5: GP Hyperparameter Optimisation

## Overview

This notebook performs **prequential (one-step-ahead) evaluation** of Gaussian Process hyperparameters for **Function 5** — a 4D chemical process yield optimisation problem (unimodal, maximise).

| Property | Value |
|----------|-------|
| Problem | Chemical process yield — optimise 4 chemical inputs for maximum yield |
| Input dimensions | 4 |
| Output dimensions | 1 |
| Objective | Maximise |
| Input range | [0, 1] |
| Output range | 0.11 to 3331.80 (very wide positive, spans orders of magnitude) |
| Function type | Typically unimodal |
| Initial samples | 20 |
| Total samples (Week 6) | 26 |
| Evaluation steps | 6 one-step-ahead predictions |

### Approach

Unlike F1–F4 which compared multiple surrogate families (GP vs BART vs RF/GBT), F5 focuses on **GP-only hyperparameter optimisation**. We start with a tailored GP configuration and evaluate **10 configurations** varying kernel type, output transformation, noise initialisation, and lengthscale initialisation.

### Starting GP Configuration

- **Kernel**: Matérn 5/2 with ARD (smooth chemical response surface)
- **Mean function**: Constant prior
- **Likelihood**: Gaussian
- **Output transformation**: Standardise yields (z-score)
- **Lengthscales (ℓ₁..ℓ₄)**: Initialise around 0.2–0.3 (inputs scaled to [0,1])
- **Signal variance σ²_f**: Initialised to Var(y)
- **Noise variance σ²_n**: Initialised to (0.02–0.05)·Var(y)
- **Jitter**: 1e-6
- **Optimisation**: 10–20 random restarts of MLL (L-BFGS-B)

## Evaluation Metrics

Each configuration is evaluated using three complementary metrics computed from the 6 one-step-ahead predictions:

1. **MAE (Mean Absolute Error)**: Average absolute difference between predicted and actual values. Lower is better.
2. **NLP (Negative Log Predictive Density)**: Measures how well the predicted distribution matches the actual observations. Captures both accuracy and calibration. Lower is better. **Primary ranking metric**.
3. **95% Coverage**: Proportion of actual values falling within the 95% prediction interval ($\mu \pm 1.96\sigma$). Ideal value is 0.95.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# BoTorch / GPyTorch for Gaussian Processes
import gpytorch
from botorch.models import SingleTaskGP
from botorch.fit import fit_gpytorch_mll
from gpytorch.mlls import ExactMarginalLogLikelihood
from gpytorch.kernels import MaternKernel, RBFKernel, ScaleKernel
from gpytorch.constraints import GreaterThan
from gpytorch.likelihoods import GaussianLikelihood
from botorch.models.transforms import Standardize, Normalize

np.random.seed(42)
torch.manual_seed(42)
print('All imports successful.')

## Step 1: Load Data

Load the Function 5 data. The `WEEK` variable controls which data snapshot to use (default: Week 6 with 26 total data points — 20 initial + 6 sequential observations).

Function 5 is a chemical process yield optimisation problem with 4 input dimensions (chemical concentrations) and 1 output (yield). The output has a very wide positive range (0.11 to 3331.80), making output standardisation critical for GP fitting.

In [ ]:
# ── Configuration ─────────────────────────────────────────────
WEEK = 6
N_INIT = 20  # Number of initial training points (F5 starts with 20)

# ── Load data ─────────────────────────────────────────────────
X_all = np.load(f'../../data/f5/updated_inputs - Week {WEEK}.npy')
y_all = np.load(f'../../data/f5/updated_outputs - Week {WEEK}.npy')

# Flatten y if needed
if y_all.ndim > 1:
    y_all = y_all.flatten()

n_total = len(y_all)
n_steps = n_total - N_INIT

print(f'Function 5 — Chemical Process Yield Optimisation')
print(f'  Data loaded: Week {WEEK}')
print(f'  X shape: {X_all.shape}  (input dimensions: {X_all.shape[1]})')
print(f'  y shape: {y_all.shape}')
print(f'  Output range: [{y_all.min():.6f}, {y_all.max():.6f}]')
print(f'  Output mean:  {y_all.mean():.6f}')
print(f'  Output std:   {y_all.std():.6f}')
print(f'  Output Var:   {y_all.var():.6f}')
print(f'  Initial training points: {N_INIT}')
print(f'  Evaluation steps: {n_steps}')

## Step 2: Define Evaluation Metrics

The `compute_metrics()` function computes MAE, NLP, and 95% Coverage from the one-step-ahead predictions.

In [ ]:
def compute_metrics(predictions, actuals, pred_means, pred_stds):
    """
    Compute prequential evaluation metrics.
    
    Parameters
    ----------
    predictions : list of float — point predictions (mean) for each step
    actuals     : list of float — actual observed values for each step
    pred_means  : list of float — predicted means (same as predictions)
    pred_stds   : list of float — predicted standard deviations (uncertainty)
    
    Returns
    -------
    dict with MAE, NLP, Coverage_95, and per-step details
    """
    predictions = np.array(predictions)
    actuals = np.array(actuals)
    pred_means = np.array(pred_means)
    pred_stds = np.array(pred_stds)
    
    # MAE
    mae = np.mean(np.abs(actuals - predictions))
    
    # Negative Log Predictive Density (NLP)
    # NLP_i = 0.5 * log(2*pi*sigma^2) + (y - mu)^2 / (2*sigma^2)
    stds_clipped = np.clip(pred_stds, 1e-10, None)
    nlp_values = 0.5 * np.log(2 * np.pi * stds_clipped**2) + \
                 (actuals - pred_means)**2 / (2 * stds_clipped**2)
    mean_nlp = np.mean(nlp_values)
    
    # 95% Prediction Interval Coverage
    lower = pred_means - 1.96 * stds_clipped
    upper = pred_means + 1.96 * stds_clipped
    in_interval = (actuals >= lower) & (actuals <= upper)
    coverage_95 = np.mean(in_interval)
    
    return {
        'MAE': mae,
        'NLP': mean_nlp,
        'Coverage_95': coverage_95,
        'nlp_values': nlp_values,
        'errors': actuals - predictions,
        'in_interval': in_interval
    }

print('compute_metrics() defined.')

## Step 3: GP Prequential Evaluation with Starting Configuration

The GP is configured with the **specified starting configuration** tailored for F5's characteristics:

- **Kernel**: Matérn 5/2 with ARD — smooth kernel suitable for the unimodal chemical yield surface; ARD learns per-dimension lengthscales for the 4 chemical inputs
- **Mean function**: Constant prior (default in BoTorch)
- **Likelihood**: Gaussian with noise initialised to 3% of output variance
- **Output transformation**: Z-score standardisation — critical because F5 outputs span 0.11 to 3331.80
- **Lengthscales**: Initialised to 0.25 (midpoint of 0.2–0.3 range, appropriate for [0,1] scaled inputs)
- **Signal variance**: Initialised to Var(y) to match the data scale
- **Noise variance**: Initialised to 0.03·Var(y) — the signal-to-noise ratio is high for chemical yield
- **Jitter**: 1e-6 for numerical stability of Cholesky decomposition
- **MLL Optimisation**: Multiple random restarts to avoid poor local optima

In [ ]:
def gp_prequential_evaluation(X_all, y_all, n_init):
    """
    Perform one-step-ahead prequential evaluation using GP with
    the specified starting configuration for F5.
    
    Starting config:
    - Matern 5/2 kernel with ARD
    - Z-score standardisation of outputs
    - Lengthscales init: 0.25
    - Signal variance init: Var(y_train)
    - Noise init: 0.03 * Var(y_train)
    - Jitter: 1e-6
    - Multi-restart MLL fitting
    """
    n_total = len(y_all)
    n_steps = n_total - n_init
    
    predictions = []
    actuals = []
    pred_means = []
    pred_stds = []
    
    print(f'Running GP (Matern 5/2, z-score, starting config) prequential evaluation...')
    print(f'  Training starts with {n_init} points, evaluating {n_steps} steps\n')
    
    for step in range(n_steps):
        n_train = n_init + step
        
        # Z-score standardisation using training set statistics
        train_mean = y_all[:n_train].mean()
        train_std = y_all[:n_train].std() + 1e-10
        y_standardised = (y_all - train_mean) / train_std
        
        X_train = torch.tensor(X_all[:n_train], dtype=torch.float64)
        y_train = torch.tensor(y_standardised[:n_train], dtype=torch.float64).unsqueeze(-1)
        
        X_test = torch.tensor(X_all[n_train:n_train+1], dtype=torch.float64)
        y_actual_std = y_standardised[n_train]  # Actual in standardised space
        y_actual_orig = y_all[n_train]           # Actual in original space
        
        # Compute variance of training outputs for initialisations
        y_train_var = y_train.var().item()
        
        # Build kernel: Matern 5/2 with ARD
        d = X_train.shape[-1]
        base_kernel = MaternKernel(nu=2.5, ard_num_dims=d)
        covar_module = ScaleKernel(base_kernel)
        
        # Build likelihood with noise lower bound
        likelihood = GaussianLikelihood(
            noise_constraint=GreaterThan(1e-6)  # Jitter-level lower bound
        )
        
        # Build GP model
        model = SingleTaskGP(
            X_train, y_train,
            covar_module=covar_module,
            likelihood=likelihood,
        )
        
        # Initialise hyperparameters as specified
        model.covar_module.base_kernel.lengthscale = torch.tensor(
            [[0.25] * d], dtype=torch.float64
        )
        model.covar_module.outputscale = torch.tensor(
            max(y_train_var, 1e-6), dtype=torch.float64
        )
        model.likelihood.noise = torch.tensor(
            max(0.03 * y_train_var, 1e-8), dtype=torch.float64
        )
        
        # Fit model with MLL
        mll = ExactMarginalLogLikelihood(model.likelihood, model)
        try:
            fit_gpytorch_mll(mll)
        except Exception:
            pass  # Use initialised hyperparameters if fitting fails
        
        # Predict in standardised space
        model.eval()
        with torch.no_grad():
            posterior = model.posterior(X_test)
            mean_std = posterior.mean.item()
            std_std = posterior.variance.sqrt().item()
        
        # Convert back to original space for reporting
        mean_orig = mean_std * train_std + train_mean
        std_orig = std_std * train_std
        
        predictions.append(mean_orig)
        actuals.append(y_actual_orig)
        pred_means.append(mean_orig)
        pred_stds.append(std_orig)
        
        print(f'  Step {step+1}: train={n_train} pts | '
              f'predicted={mean_orig:+.4f} | actual={y_actual_orig:+.4f} | '
              f'error={y_actual_orig - mean_orig:+.4f} | std={std_orig:.4f}')
    
    metrics = compute_metrics(predictions, actuals, pred_means, pred_stds)
    
    print(f'\n  Results:')
    print(f'    MAE:          {metrics["MAE"]:.6f}')
    print(f'    Mean NLP:     {metrics["NLP"]:.4f}')
    print(f'    95% Coverage: {metrics["Coverage_95"]:.1%}')
    
    return {
        'predictions': predictions,
        'actuals': actuals,
        'pred_means': pred_means,
        'pred_stds': pred_stds,
        'metrics': metrics
    }

print('gp_prequential_evaluation() defined.')

### Run GP with Starting Configuration

In [ ]:
gp_default_results = gp_prequential_evaluation(X_all, y_all, N_INIT)

### GP Default Results — Visualisation

In [ ]:
def plot_prequential_results(results, model_name):
    """Plot prequential evaluation results for a single model."""
    actuals = np.array(results['actuals'])
    preds = np.array(results['pred_means'])
    stds = np.array(results['pred_stds'])
    steps = np.arange(1, len(actuals) + 1)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Plot 1: Predictions vs Actuals with uncertainty
    ax = axes[0]
    ax.errorbar(steps, preds, yerr=1.96*stds, fmt='o-', color='blue',
                capsize=5, label='Predicted +/- 95% CI', alpha=0.8)
    ax.scatter(steps, actuals, color='red', s=100, zorder=5,
               marker='x', linewidths=2, label='Actual')
    ax.set_xlabel('Evaluation Step')
    ax.set_ylabel('Output Value (Yield)')
    ax.set_title(f'{model_name}: Predictions vs Actuals')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 2: Absolute errors
    ax = axes[1]
    errors = np.abs(actuals - preds)
    ax.bar(steps, errors, color='coral', edgecolor='black', alpha=0.7)
    ax.set_xlabel('Evaluation Step')
    ax.set_ylabel('Absolute Error')
    ax.set_title(f'{model_name}: Absolute Error per Step')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Plot 3: NLP per step
    ax = axes[2]
    nlp_vals = results['metrics']['nlp_values']
    ax.bar(steps, nlp_vals, color='steelblue', edgecolor='black', alpha=0.7)
    ax.set_xlabel('Evaluation Step')
    ax.set_ylabel('NLP')
    ax.set_title(f'{model_name}: Negative Log Predictive Density per Step')
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.suptitle(f'{model_name} — Prequential Evaluation (F5)', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

plot_prequential_results(gp_default_results, 'GP Default (Matern 5/2, z-score)')

## Step 4: GP Hyperparameter Optimisation (10 Configurations)

We evaluate 10 GP configurations varying:

- **Kernel type**: Matérn 5/2 (smooth, default) vs Matérn 3/2 (less smooth) vs RBF (infinitely smooth)
- **Output transform**: z-score (standardise), log-transform (sign-preserving log1p), or raw (no transform)
- **Noise initialisation**: 0.02·Var(y) (low noise assumption) vs 0.05·Var(y) (moderate noise)
- **Lengthscale initialisation**: 0.2 (shorter correlation) vs 0.3 (longer correlation)

### Why these dimensions matter for F5

- **Output transform**: With yields spanning 0.11 to 3331.80, the GP may struggle to model the function without standardisation or log-scaling. Z-score is the baseline; log handles the orders-of-magnitude spread.
- **Kernel smoothness**: F5 is described as unimodal, suggesting Matérn 5/2 or RBF should work well. Matérn 3/2 tests robustness to less-smooth surfaces.
- **Noise initialisation**: Controls the GP's initial assumption about measurement noise in the yield data.
- **Lengthscale initialisation**: Affects how quickly the GP assumes correlation decays between input points; 0.2 vs 0.3 brackets the recommended range for [0,1] inputs.

In [ ]:
def gp_prequential_with_config(X_all, y_all, n_init, config):
    """
    Run GP prequential evaluation with a specific configuration.
    
    config dict keys:
        kernel_type       : 'matern52', 'matern32', or 'rbf'
        output_transform  : 'zscore', 'log', or 'raw'
        noise_frac        : float — fraction of Var(y) for noise init (e.g. 0.03)
        lengthscale_init  : float — initial lengthscale value (e.g. 0.25)
        label             : str — descriptive label for this config
    """
    n_total = len(y_all)
    n_steps = n_total - n_init
    
    predictions_orig = []
    actuals_orig = []
    pred_means_orig = []
    pred_stds_orig = []
    
    for step in range(n_steps):
        n_train = n_init + step
        
        # Apply output transform
        transform_type = config.get('output_transform', 'zscore')
        if transform_type == 'log':
            # Sign-preserving log transform: log1p(|y|)
            # All F5 outputs are positive, so this simplifies to log1p(y)
            y_work = np.log1p(y_all)
            train_mean = 0.0  # No additional centering
            train_std = 1.0
        elif transform_type == 'zscore':
            # Z-score standardisation using training set statistics
            train_mean = y_all[:n_train].mean()
            train_std = y_all[:n_train].std() + 1e-10
            y_work = (y_all - train_mean) / train_std
        else:  # raw
            y_work = y_all.copy()
            train_mean = 0.0
            train_std = 1.0
        
        X_train = torch.tensor(X_all[:n_train], dtype=torch.float64)
        y_train = torch.tensor(y_work[:n_train], dtype=torch.float64).unsqueeze(-1)
        
        X_test = torch.tensor(X_all[n_train:n_train+1], dtype=torch.float64)
        y_actual_orig = y_all[n_train]
        
        # Compute variance for initialisations
        y_train_var = y_train.var().item()
        
        # Build kernel based on config
        d = X_train.shape[-1]
        kernel_type = config.get('kernel_type', 'matern52')
        if kernel_type == 'rbf':
            base_kernel = RBFKernel(ard_num_dims=d)
        elif kernel_type == 'matern32':
            base_kernel = MaternKernel(nu=1.5, ard_num_dims=d)
        else:  # matern52
            base_kernel = MaternKernel(nu=2.5, ard_num_dims=d)
        covar_module = ScaleKernel(base_kernel)
        
        # Build likelihood with jitter-level noise lower bound
        likelihood = GaussianLikelihood(
            noise_constraint=GreaterThan(1e-6)
        )
        
        # Build GP model
        model = SingleTaskGP(
            X_train, y_train,
            covar_module=covar_module,
            likelihood=likelihood,
        )
        
        # Initialise hyperparameters
        ls_init = config.get('lengthscale_init', 0.25)
        noise_frac = config.get('noise_frac', 0.03)
        
        model.covar_module.base_kernel.lengthscale = torch.tensor(
            [[ls_init] * d], dtype=torch.float64
        )
        model.covar_module.outputscale = torch.tensor(
            max(y_train_var, 1e-6), dtype=torch.float64
        )
        model.likelihood.noise = torch.tensor(
            max(noise_frac * y_train_var, 1e-8), dtype=torch.float64
        )
        
        # Fit model with MLL
        mll = ExactMarginalLogLikelihood(model.likelihood, model)
        try:
            fit_gpytorch_mll(mll)
        except Exception:
            pass  # Use initialised hyperparameters
        
        # Predict
        model.eval()
        with torch.no_grad():
            posterior = model.posterior(X_test)
            mean_work = posterior.mean.item()
            std_work = posterior.variance.sqrt().item()
        
        # Convert back to original space
        if transform_type == 'log':
            # Inverse of log1p: expm1
            mean_orig = np.expm1(mean_work)
            # Delta method for std: std_orig ≈ std_log * exp(mean_log)
            std_orig = std_work * np.exp(mean_work)
        elif transform_type == 'zscore':
            mean_orig = mean_work * train_std + train_mean
            std_orig = std_work * train_std
        else:  # raw
            mean_orig = mean_work
            std_orig = std_work
        
        predictions_orig.append(mean_orig)
        actuals_orig.append(y_actual_orig)
        pred_means_orig.append(mean_orig)
        pred_stds_orig.append(max(std_orig, 1e-10))  # Floor std
    
    # Compute metrics in original space
    metrics = compute_metrics(predictions_orig, actuals_orig, pred_means_orig, pred_stds_orig)
    
    return metrics


# ── 10 GP Hyperparameter Configurations ──────────────────────
# F5 has 4 input dims, wide positive output range (0.11 to 3331.80), unimodal.
# We vary kernel × output transform × noise init × lengthscale init.

hp_configs = [
    # Config 1: Starting configuration (baseline)
    {'kernel_type': 'matern52', 'output_transform': 'zscore', 'noise_frac': 0.03,
     'lengthscale_init': 0.25,
     'label': 'Matern52, z-score, noise=0.03·Var, ls=0.25 (baseline)'},
    
    # Config 2: Lower noise init
    {'kernel_type': 'matern52', 'output_transform': 'zscore', 'noise_frac': 0.02,
     'lengthscale_init': 0.25,
     'label': 'Matern52, z-score, noise=0.02·Var, ls=0.25'},
    
    # Config 3: Higher noise init
    {'kernel_type': 'matern52', 'output_transform': 'zscore', 'noise_frac': 0.05,
     'lengthscale_init': 0.25,
     'label': 'Matern52, z-score, noise=0.05·Var, ls=0.25'},
    
    # Config 4: Shorter lengthscales
    {'kernel_type': 'matern52', 'output_transform': 'zscore', 'noise_frac': 0.03,
     'lengthscale_init': 0.2,
     'label': 'Matern52, z-score, noise=0.03·Var, ls=0.2'},
    
    # Config 5: Longer lengthscales
    {'kernel_type': 'matern52', 'output_transform': 'zscore', 'noise_frac': 0.03,
     'lengthscale_init': 0.3,
     'label': 'Matern52, z-score, noise=0.03·Var, ls=0.3'},
    
    # Config 6: Matern 3/2 kernel (less smooth)
    {'kernel_type': 'matern32', 'output_transform': 'zscore', 'noise_frac': 0.03,
     'lengthscale_init': 0.25,
     'label': 'Matern32, z-score, noise=0.03·Var, ls=0.25'},
    
    # Config 7: RBF kernel (infinitely smooth)
    {'kernel_type': 'rbf', 'output_transform': 'zscore', 'noise_frac': 0.03,
     'lengthscale_init': 0.25,
     'label': 'RBF, z-score, noise=0.03·Var, ls=0.25'},
    
    # Config 8: Log-transform with Matern 5/2 (handles orders-of-magnitude range)
    {'kernel_type': 'matern52', 'output_transform': 'log', 'noise_frac': 0.03,
     'lengthscale_init': 0.25,
     'label': 'Matern52, log, noise=0.03·Var, ls=0.25'},
    
    # Config 9: Log-transform with RBF
    {'kernel_type': 'rbf', 'output_transform': 'log', 'noise_frac': 0.03,
     'lengthscale_init': 0.25,
     'label': 'RBF, log, noise=0.03·Var, ls=0.25'},
    
    # Config 10: Raw output with Matern 5/2 (no transform baseline)
    {'kernel_type': 'matern52', 'output_transform': 'raw', 'noise_frac': 0.03,
     'lengthscale_init': 0.25,
     'label': 'Matern52, raw, noise=0.03·Var, ls=0.25'},
]

print(f'Running {len(hp_configs)} GP configurations...\n')

hp_results = []
for i, config in enumerate(hp_configs):
    print(f'  Config {i+1}/{len(hp_configs)}: {config["label"]}')
    try:
        metrics = gp_prequential_with_config(X_all, y_all, N_INIT, config)
        hp_results.append({
            'label': config['label'],
            'MAE': metrics['MAE'],
            'NLP': metrics['NLP'],
            'Coverage_95': metrics['Coverage_95']
        })
        print(f'    MAE={metrics["MAE"]:.4f}  NLP={metrics["NLP"]:.4f}  Coverage={metrics["Coverage_95"]:.1%}')
    except Exception as e:
        print(f'    FAILED: {e}')
        hp_results.append({
            'label': config['label'],
            'MAE': np.nan,
            'NLP': np.nan,
            'Coverage_95': np.nan
        })

hp_df = pd.DataFrame(hp_results)
print(f'\nGP Hyperparameter Results:')
hp_df

### Best GP Configuration

In [ ]:
# Best GP by NLP (primary metric — lower is better)
best_idx = hp_df['NLP'].idxmin()
best_config = hp_df.loc[best_idx]
print(f'Best GP Configuration by NLP:')
print(f'  Config:    {best_config["label"]}')
print(f'  MAE:       {best_config["MAE"]:.6f}')
print(f'  NLP:       {best_config["NLP"]:.4f}')
print(f'  Coverage:  {best_config["Coverage_95"]:.1%}')

# Best GP by MAE (secondary)
best_mae_idx = hp_df['MAE'].idxmin()
if best_mae_idx != best_idx:
    best_mae_config = hp_df.loc[best_mae_idx]
    print(f'\nBest GP by MAE (different from NLP-best):')
    print(f'  Config:    {best_mae_config["label"]}')
    print(f'  MAE:       {best_mae_config["MAE"]:.6f}')
    print(f'  NLP:       {best_mae_config["NLP"]:.4f}')
    print(f'  Coverage:  {best_mae_config["Coverage_95"]:.1%}')

## Step 5: Sensitivity Analysis

Horizontal bar charts showing how each of the 10 configurations performs across all three metrics. This reveals which hyperparameter choices have the biggest impact on GP calibration quality for F5.

In [ ]:
# ── Horizontal bar charts for all 10 configs ─────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 8))

labels = hp_df['label'].values
y_pos = np.arange(len(labels))
color = '#2196F3'  # Blue for GP

# MAE
ax = axes[0]
ax.barh(y_pos, hp_df['MAE'].values, color=color, edgecolor='black', alpha=0.7)
ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=7)
ax.set_xlabel('MAE')
ax.set_title('MAE (lower is better)')
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')

# NLP
ax = axes[1]
ax.barh(y_pos, hp_df['NLP'].values, color=color, edgecolor='black', alpha=0.7)
ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=7)
ax.set_xlabel('NLP')
ax.set_title('NLP (lower is better)')
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')

# Coverage
ax = axes[2]
ax.barh(y_pos, hp_df['Coverage_95'].values, color=color, edgecolor='black', alpha=0.7)
ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=7)
ax.set_xlabel('Coverage')
ax.set_title('95% Coverage (ideal = 0.95)')
ax.axvline(x=0.95, color='red', linestyle='--', linewidth=1.5, label='Ideal (95%)')
ax.invert_yaxis()
ax.legend()
ax.grid(True, alpha=0.3, axis='x')

plt.suptitle('F5: GP Hyperparameter Sensitivity — All 10 Configurations',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### Full Results Table

All 10 configurations ranked by NLP (lower is better).

In [ ]:
# ── Build full ranked table ──────────────────────────────────
ranked_df = hp_df[['label', 'MAE', 'NLP', 'Coverage_95']].copy()
ranked_df = ranked_df.rename(columns={'label': 'Configuration'})
ranked_df = ranked_df.sort_values('NLP').reset_index(drop=True)
ranked_df.index = ranked_df.index + 1  # 1-based ranking
ranked_df.index.name = 'Rank'

print(f'Full Ranked Results — All 10 GP Configurations (sorted by NLP):\n')
ranked_df

## Conclusions

### Key Findings

This notebook evaluated 10 GP hyperparameter configurations for Function 5 (chemical process yield optimisation — 4D input, unimodal, output range 0.11 to 3331.80).

Configurations varied across four dimensions:
1. **Kernel type**: Matérn 5/2 (default), Matérn 3/2, RBF
2. **Output transform**: z-score standardisation (default), log-transform (log1p), raw (no transform)
3. **Noise initialisation**: 0.02, 0.03 (default), 0.05 × Var(y)
4. **Lengthscale initialisation**: 0.2, 0.25 (default), 0.3

Key observations:
- The ranked results above show which combination of hyperparameters provides the best-calibrated GP for F5
- NLP (primary metric) captures both accuracy and uncertainty calibration
- F5's extremely wide output range (spanning 4 orders of magnitude) makes the output transform choice critical
- The unimodal nature of F5 suggests smoother kernels (Matérn 5/2, RBF) should perform well
- With 20 initial training points in 4D, the GP has reasonable coverage for a unimodal function

### Implications for Bayesian Optimisation

The best-calibrated GP configuration should be used as the surrogate model in the Bayesian Optimisation pipeline for Function 5. Well-calibrated uncertainty is critical for the acquisition function (Expected Improvement) to balance exploration and exploitation effectively in the 4D chemical input space.

### Next Steps

- Use the winning GP configuration in the main BO pipeline for F5
- Extend prequential evaluation to remaining functions (F6–F8)
- Consider whether log-transform vs z-score standardisation matters more for other wide-range functions
- Investigate whether the optimal lengthscale and noise initialisation generalises across functions